# Phase 5: Offline Evaluation of Causal-RL Agent

Evaluates the trained PPO agent from Phase 4 on held-out test sessions (MIND-small).
Compares against Random and Popularity baselines using NDCG@K, Precision@K, and ILD.

## Scope Lock
- Phase 5 only: offline replay evaluation on `scm_test.parquet`.
- Uses `replay_evaluate()` from `src.evaluation.metrics`.
- Baselines: random ranking, popularity-based ranking.
- Significance testing via paired t-test.

## Inputs
- `data/scm_test.parquet` — held-out test impressions
- `artifacts/checkpoints/ppo_causal_rs_w03.zip` — trained PPO policy

## Outputs
- Per-metric comparison table
- Significance test results

In [ ]:
import warnings
import logging
import pickle
import sys
import time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from scipy import stats
import torch

_cwd = Path.cwd()
_root = _cwd.parent if _cwd.name == "notebooks" else _cwd
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src.evaluation.metrics import (
    ndcg_at_k, precision_at_k, ild,
    compute_session_metrics, aggregate_metrics, significance_test,
)
from src.rl_agent.environment import NewsRecommendEnv

from stable_baselines3 import PPO

warnings.filterwarnings('ignore')
logging.getLogger('stable_baselines3').setLevel(logging.WARNING)

DATA = _root / "data"
ARTIFACTS = _root / "artifacts"
MODEL_PATH = ARTIFACTS / "checkpoints" / "ppo_causal_rs_w03"
print(f"Project root: {_root}")
print(f"Model path: {MODEL_PATH}")

## 2. Load Data
Load test data, news lookup, and CDI cache.

In [ ]:
test_df = pd.read_parquet(DATA / "scm_test.parquet")
print(f"Test data: {len(test_df)} rows, {test_df['user_id'].nunique()} users, {test_df['impression_id'].nunique()} impressions")

train_df = pd.read_parquet(DATA / "scm_train.parquet")
print(f"Train data: {len(train_df)} rows (for popularity baseline)")

In [ ]:
cdi_path = ARTIFACTS / "cdi_cache.pkl"
if cdi_path.exists():
    with open(cdi_path, "rb") as f:
        cdi_cache = pickle.load(f)
    print(f"CDI cache: {len(cdi_cache)} entries")
else:
    cdi_cache = {}
    print("No CDI cache found — using zeros for CDI reward")

## 3. Build Test Sessions
Create session objects for offline replay, one per test impression.

In [ ]:
def build_sessions(df):
    sessions = []
    for imp_id, group in df.groupby("impression_id", sort=False):
        user_id = group["user_id"].iloc[0]
        history_emb = group["U_history_emb_full"].iloc[0]
        if isinstance(history_emb, np.ndarray):
            pass
        elif isinstance(history_emb, (list, str)):
            history_emb = np.array(eval(history_emb) if isinstance(history_emb, str) else history_emb, dtype=np.float32)
        else:
            history_emb = np.frombuffer(history_emb, dtype=np.float32)
        history_emb = np.asarray(history_emb, dtype=np.float32).flatten()

        item_ids = group["item_id"].tolist()
        clicks = group["Y_click"].tolist()
        title_embs = []
        for emb in group["I_title_emb_full"]:
            if isinstance(emb, np.ndarray):
                title_embs.append(emb)
            elif isinstance(emb, (list, str)):
                title_embs.append(np.array(eval(emb) if isinstance(emb, str) else emb, dtype=np.float32))
            else:
                title_embs.append(np.frombuffer(emb, dtype=np.float32))

        clicked_items = set(group[group["Y_click"] == 1]["item_id"].tolist())

        candidates = [
            type("C", (), {"item_id": iid, "title_emb": emb})()
            for iid, emb in zip(item_ids, title_embs)
        ]
        session = type("Session", (), {
            "user_id": user_id,
            "initial_history_emb": history_emb,
            "candidates": [candidates],
            "clicks": [clicks],
            "candidate_pool": item_ids,
            "clicked_items": clicked_items,
        })()
        sessions.append(session)
    return sessions

test_sessions = build_sessions(test_df)
print(f"Built {len(test_sessions)} test sessions")

## 4. Build News Lookup DataFrame
Build a mapping from item_id to I_title_emb_full for ILD computation.

In [ ]:
news_df = test_df[["item_id", "I_title_emb_full", "I_category"]].drop_duplicates("item_id")
news_df = news_df.set_index("item_id")
print(f"News lookup: {len(news_df)} unique items")

## 5. Load Trained PPO Model
Load the PPO checkpoint from Phase 4.

In [ ]:
if MODEL_PATH.with_suffix(".zip").exists():
    model = PPO.load(str(MODEL_PATH), device="cpu")
    print(f"PPO model loaded from {MODEL_PATH}.zip")
    print(f"Policy: {model.policy.__class__.__name__}")
    print(f"Observation space: {model.observation_space}")
    print(f"Action space: {model.action_space}")
else:
    print(f"Model not found at {MODEL_PATH}.zip — will use random policy only")
    model = None

## 6. Evaluate Trained PPO Agent
Run offline replay on test sessions using the trained policy.

In [ ]:
K = 10
T = 1

def evaluate_policy(policy, sessions, news_df, cdi_cache, w=0.6, K=K, T=T, label="policy"):
    """Evaluate a policy on test sessions and return per-session metrics."""
    results = []
    for i, session in enumerate(sessions):
        if policy is None:
            ranked = list(range(len(session.candidate_pool)))
            np.random.shuffle(ranked)
            rec_items = [session.candidate_pool[a] for a in ranked[:K]]
        else:
            K_train = int(policy.action_space.n)
            pool = session.candidate_pool
            if len(pool) > K_train:
                chosen = list(np.random.choice(len(pool), K_train, replace=False))
            else:
                chosen = list(range(len(pool)))
            cand_items = [pool[a] for a in chosen]
            cand_embs = [session.candidates[0][a].title_emb for a in chosen]
            cand_clicks = [session.clicks[0][a] for a in chosen]
            mock_sess = type("Session", (), {
                "user_id": session.user_id,
                "initial_history_emb": session.initial_history_emb,
                "candidates": [[type("C", (), {"item_id": iid, "title_emb": emb})() for iid, emb in zip(cand_items, cand_embs)]],
                "clicks": [cand_clicks],
                "candidate_pool": cand_items,
                "clicked_items": session.clicked_items,
            })()
            env = NewsRecommendEnv([mock_sess], news_df, cdi_cache, w=w, K=K_train, T=T)
            obs, _ = env.reset()
            import torch
            with torch.no_grad():
                dist = policy.policy.get_distribution(torch.as_tensor(obs[None], dtype=torch.float32))
                logits = dist.distribution.logits[0]
            ranked = np.argsort(logits.detach().cpu().numpy())[::-1]
            rec_items = [cand_items[a] for a in ranked[:K]]
            env.close()

        rec_embs = []
        for iid in rec_items:
            try:
                emb = news_df.loc[iid, "I_title_emb_full"]
                if isinstance(emb, np.ndarray):
                    rec_embs.append(emb)
                elif isinstance(emb, (list, str)):
                    rec_embs.append(np.array(eval(emb) if isinstance(emb, str) else emb, dtype=np.float32))
            except KeyError:
                rec_embs.append(np.zeros(768, dtype=np.float32))

        ndcg = ndcg_at_k(rec_items, session.clicked_items, K)
        prec = precision_at_k(rec_items, session.clicked_items, K)
        ild_val = ild(np.array(rec_embs)) if len(rec_embs) >= 2 else 0.0

        results.append({"ndcg": ndcg, "precision": prec, "ild": ild_val})

        if (i + 1) % 100 == 0:
            interim = pd.DataFrame(results)
            print(f"  [{label}] {i+1}/{len(sessions)} � NDCG: {interim['ndcg'].mean():.4f}, Prec: {interim['precision'].mean():.4f}, ILD: {interim['ild'].mean():.4f}")

    return results

print("Evaluating PPO agent on test sessions...")
t0 = time.time()
ppo_results = evaluate_policy(model, test_sessions, news_df, cdi_cache, label="PPO")
t_ppo = time.time() - t0
print(f"PPO evaluation completed in {t_ppo:.1f}s")

## 7. Baseline: Random Ranking
Rank candidate items randomly.

In [ ]:
print("Evaluating Random baseline...")
t0 = time.time()
rand_results = evaluate_policy(None, test_sessions, news_df, cdi_cache, label="Random")
t_rand = time.time() - t0
print(f"Random evaluation completed in {t_rand:.1f}s")

## 8. Baseline: Popularity
Rank by item frequency in the training set (higher frequency = higher rank).

In [ ]:
pop_counter = Counter(train_df["item_id"])
print(f"Popularity baseline: {len(pop_counter)} unique items, max freq={pop_counter.most_common(1)[0][1]}")

pop_results = []
for i, session in enumerate(test_sessions):
    scored = [(iid, pop_counter.get(iid, 0)) for iid in session.candidate_pool]
    scored.sort(key=lambda x: -x[1])
    rec_items = [s[0] for s in scored[:K]]

    rec_embs = []
    for iid in rec_items:
        try:
            emb = news_df.loc[iid, "I_title_emb_full"]
            if isinstance(emb, np.ndarray):
                rec_embs.append(emb)
            elif isinstance(emb, (list, str)):
                rec_embs.append(np.array(eval(emb) if isinstance(emb, str) else emb, dtype=np.float32))
        except KeyError:
            rec_embs.append(np.zeros(768, dtype=np.float32))

    ndcg = ndcg_at_k(rec_items, session.clicked_items, K)
    prec = precision_at_k(rec_items, session.clicked_items, K)
    ild_val = ild(np.array(rec_embs)) if len(rec_embs) >= 2 else 0.0
    pop_results.append({"ndcg": ndcg, "precision": prec, "ild": ild_val})

    if (i + 1) % 100 == 0:
        interim = pd.DataFrame(pop_results)
        print(f"  [Popularity] {i+1}/{len(test_sessions)} — NDCG: {interim['ndcg'].mean():.4f}, Prec: {interim['precision'].mean():.4f}, ILD: {interim['ild'].mean():.4f}")

print("Popularity evaluation done.")

## 9. Results Summary
Aggregate metrics and pairwise significance tests.

In [ ]:
def summarize(label, results):
    df = pd.DataFrame(results)
    return {
        "Method": label,
        "NDCG@K_mean": f"{df['ndcg'].mean():.4f}",
        "NDCG@K_std": f"{df['ndcg'].std():.4f}",
        "Precision@K_mean": f"{df['precision'].mean():.4f}",
        "Precision@K_std": f"{df['precision'].std():.4f}",
        "ILD_mean": f"{df['ild'].mean():.4f}",
        "ILD_std": f"{df['ild'].std():.4f}",
        "n_sessions": len(results),
    }

summaries = []
if ppo_results:
    summaries.append(summarize("PPO (Causal-RL)", ppo_results))
summaries.append(summarize("Random", rand_results))
summaries.append(summarize("Popularity", pop_results))

summary_df = pd.DataFrame(summaries)
print("### Aggregated Metrics")
print(summary_df.to_string(index=False))

print("\n### Significance Tests (paired t-test vs Random)")
if ppo_results:
    ppo_df = pd.DataFrame(ppo_results)
    rand_df = pd.DataFrame(rand_results)
    pop_df = pd.DataFrame(pop_results)

    for metric in ["ndcg", "precision", "ild"]:
        print(f"\n--- {metric.upper()} ---")
        t_stat, p_val = stats.ttest_rel(ppo_df[metric], rand_df[metric])
        d = (ppo_df[metric].mean() - rand_df[metric].mean()) / (ppo_df[metric].std() + 1e-10)
        print(f"  PPO vs Random: t={t_stat:.3f}, p={p_val:.4f}, Cohen's d={d:.3f}")

        t_stat, p_val = stats.ttest_rel(ppo_df[metric], pop_df[metric])
        d = (ppo_df[metric].mean() - pop_df[metric].mean()) / (ppo_df[metric].std() + 1e-10)
        print(f"  PPO vs Popularity: t={t_stat:.3f}, p={p_val:.4f}, Cohen's d={d:.3f}")

print("\nPhase 5 evaluation complete.")